In [ ]:
## Model Training

"""
Python Code Autocomplete — Dual Model Training
  • TokenModel  : next-token prediction (word / punctuation boundary)
  • LineModel   : full-line completion
Includes live metric visualisation, best-model checkpointing, and an
interactive hand-test REPL.
"""
%tb
import os, json, math, random, glob
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from collections import defaultdict
from MetricsAndVisualisation import MetricLog, plot_metrics
from BestModelSaver import BestModelSaver

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

import matplotlib
matplotlib.use("Agg")           # headless — saves PNG files
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator

TOKEN_MODEL_NAME = 'token_model'
LINE_MODEL_NAME = "line_model"


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"Версия CUDA, под которую собран PyTorch: {torch.version.cuda}")
print(f"Количество GPU: {torch.cuda.device_count()}")


# ─────────────────────────────────────────────────────────────
# 1.  TOKENISER  (character-level BPE-lite, no dependencies)
# ─────────────────────────────────────────────────────────────
SPECIAL = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3}


class CodeTokenizer:
    """
    Simple sub-word tokenizer tailored for Python source code.
    Splits on whitespace/punctuation, keeps indentation tokens,
    and falls back to characters for unknowns.
    """
    PUNCT = set(",;&|~^@#")

    def __init__(self, vocab_size: int = 8000):
        self.vocab_size = vocab_size
        self.token2id: Dict[str, int] = dict(SPECIAL)
        self.id2token: Dict[int, str] = {v: k for k, v in SPECIAL.items()}
        self.built = False

    # ── build ──────────────────────────────────────────────
    def build(self, texts: List[str], min_freq: int = 3):
        freq: Dict[str, int] = defaultdict(int)
        for t in texts:
            for tok in self._raw_split(t):
                freq[tok] += 1
        sorted_tokens = sorted(freq.items(), key=lambda x: -x[1])
        for tok, cnt in sorted_tokens:
            if cnt < min_freq:
                break
            if tok not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[tok] = idx
                self.id2token[idx] = tok
        # fill remaining slots with single chars
        for c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.,_ \t\n":  #for c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_ \t\n":
            if c not in self.token2id and len(self.token2id) < self.vocab_size:
                idx = len(self.token2id)
                self.token2id[c] = idx
                self.id2token[idx] = c
        self.built = True
        print(f"[Tokenizer] vocab_size={len(self.token2id)}")

    def _raw_split(self, text: str) -> List[str]:
        tokens = []
        for line in text.splitlines(keepends=True):
            # capture leading whitespace as indent token
            stripped = line.lstrip(" \t")
            indent = line[: len(line) - len(stripped)]
            for ch in indent:
                tokens.append(ch)
            # split remainder on punctuation / spaces
            buf = ""
            for ch in stripped:
                if ch in self.PUNCT or ch in " \t\n\r":
                    if ch == "\n" or ch.strip():
                        if buf:
                            tokens.append(buf)
                        tokens.append(ch)
                        continue
                    if buf:
                        buf += ch
                        tokens.append(buf)
                        buf = ""
                        tokens.append("\n")
                else:
                    buf += ch
            if buf:
                tokens.append(buf)
        return tokens

    def encode(self, text: str) -> List[int]:
        ids = [SPECIAL["<BOS>"]]
        for tok in self._raw_split(text):
            if tok in self.token2id:
                ids.append(self.token2id[tok])
            else:
                # char fallback
                for ch in tok:
                    ids.append(self.token2id.get(ch, SPECIAL["<UNK>"]))
        ids.append(SPECIAL["<EOS>"])
        return ids

    def decode(self, ids: List[int]) -> str:
        parts = []
        for i in ids:
            tok = self.id2token.get(i, "")
            if tok in SPECIAL:
                continue
            parts.append(tok)
        return "".join(parts)

    def save(self, path: str):
        with open(path, "w") as f:
            json.dump({"token2id": self.token2id}, f)

    @classmethod
    def load(cls, path: str) -> "CodeTokenizer":
        with open(path) as f:
            d = json.load(f)
        obj = cls()
        obj.token2id = {k: int(v) for k, v in d["token2id"].items()}
        obj.id2token = {v: k for k, v in obj.token2id.items()}
        obj.built = True
        return obj

    @property
    def pad_id(self):  return SPECIAL["<PAD>"]
    @property
    def eos_id(self):  return SPECIAL["<EOS>"]
    @property
    def bos_id(self):  return SPECIAL["<BOS>"]
    @property
    def vocab(self):   return len(self.token2id)


# ─────────────────────────────────────────────────────────────
# 2.  DATASETS
# ─────────────────────────────────────────────────────────────

def load_files(data_dir: str, max_files: int = 0) -> List[str]:
    """Load .py / .txt files from a directory tree."""
    patterns = ["**/*.py", "**/*.txt"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(data_dir, pat), recursive=True))
    if max_files:
        files = files[:max_files]
    texts = []
    for fp in files:
        try:
            texts.append(Path(fp).read_text(errors="replace"))
        except Exception:
            pass
    print(f"[Data] loaded {len(texts)} files from {data_dir}")
    return texts


class TokenDataset(Dataset):
    """
    Sliding-window dataset for next-token prediction.
    Target at each position is the next token id.
    """
    def __init__(self, ids: List[int], ctx: int = 128):
        self.ctx = ctx
        self.data = torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return max(0, len(self.data) - self.ctx - 1)

    def __getitem__(self, i):
        x = self.data[i: i + self.ctx]
        y = self.data[i + 1: i + self.ctx + 1]
        return x, y


class LineDataset(Dataset):
    """
    One sample = (prefix_tokens, full_line_tokens).
    The model learns to predict the rest of the current line given a prefix.
    """
    def __init__(self, texts: List[str], tokenizer: CodeTokenizer,
                 max_prefix: int = 96, max_line: int = 64):
        self.samples: List[Tuple[List[int], List[int]]] = []
        print(len([line for text in texts for line in text.splitlines()]))
        for text in texts:
            for line in text.splitlines():
                commentary_pos = line.find('#') 
                if commentary_pos != -1 and not line[commentary_pos - 1] in ['\'', '"']:
                    # print(line)
                    line = line[:line.find('#')]
                line = line.rstrip()
                if len(line.strip()) < 10:
                    continue
                full = tokenizer.encode(line)
                if len(full) < 4:
                    continue
                split = random.randint(2, max(2, len(full) - 2))
                prefix = full[:split][-max_prefix:]
                target = full[split:][:max_line]
                target.append(tokenizer.eos_id)
                self.samples.append((prefix, target))
        print(f"[LineDataset] {len(self.samples)} samples")

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        return self.samples[i]


def collate_line(batch, pad_id: int):
    prefixes, targets = zip(*batch)
    max_p = max(len(p) for p in prefixes)
    max_t = max(len(t) for t in targets)
    P = torch.full((len(batch), max_p), pad_id, dtype=torch.long)
    T = torch.full((len(batch), max_t), pad_id, dtype=torch.long)
    for i, (p, t) in enumerate(zip(prefixes, targets)):
        P[i, :len(p)] = torch.tensor(p)
        T[i, :len(t)] = torch.tensor(t)
    return P, T


# ─────────────────────────────────────────────────────────────
# 3.  MODELS
# ─────────────────────────────────────────────────────────────

@dataclass
class ModelCfg:
    vocab: int = 8000
    d_model: int = 256
    n_heads: int = 8
    n_layers: int = 4
    d_ff: int = 1024
    max_len: int = 256
    dropout: float = 0.1


class PositionalEncoding(nn.Module):
    def __init__(self, d: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])


class TokenModel(nn.Module):
    """
    Decoder-only Transformer for causal next-token prediction.
    """
    def __init__(self, cfg: ModelCfg):
        super().__init__()
        self.cfg = cfg
        self.emb   = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
        self.pos   = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
        layer      = nn.TransformerEncoderLayer(
            cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout,
            batch_first=True, norm_first=True
        )
        self.enc   = nn.TransformerEncoder(layer, cfg.n_layers)
        self.head  = nn.Linear(cfg.d_model, cfg.vocab, bias=False)
        self.emb.weight = self.head.weight  # weight tying

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        T = x.size(1)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        h = self.pos(self.emb(x))
        h = self.enc(h, mask=mask, is_causal=True)
        return self.head(h)

    @torch.no_grad()
    def generate(self, prefix_ids: List[int], max_new: int,
                 temperature: float = 0.8, top_k: int = 50,
                 stop_at_word_end: bool = True,
                 tokenizer: Optional[CodeTokenizer] = None) -> List[int]:
        self.eval()
        dev   = next(self.parameters()).device
        ids   = list(prefix_ids)
        generated = []
        PUNCT_CHARS = set("()[]{}.,;:=+-*/\\%<>!&|~^@# \t\n\"'`")
        for _ in range(max_new):
            x = torch.tensor([ids[-self.cfg.max_len:]], dtype=torch.long, device=dev)
            logits = self(x)[0, -1] / temperature
            if top_k:
                topk_v, _ = torch.topk(logits, top_k)
                logits[logits < topk_v[-1]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            if nxt == SPECIAL["<EOS>"]:
                break
            ids.append(nxt)
            generated.append(nxt)
            if stop_at_word_end and tokenizer:
                tok = tokenizer.id2token.get(nxt, "")
                if any(c in PUNCT_CHARS for c in tok):
                    break
        return generated


class LineModel(nn.Module):
    """
    Encoder-Decoder Transformer for seq2seq line completion.
    Encoder: reads prefix.  Decoder: generates rest of line.
    """
    def __init__(self, cfg: ModelCfg):
        super().__init__()
        self.cfg = cfg
        self.enc_emb  = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
        self.dec_emb  = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
        self.enc_pos  = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
        self.dec_pos  = PositionalEncoding(cfg.d_model, cfg.max_len, cfg.dropout)
        self.transformer = nn.Transformer(
            cfg.d_model, cfg.n_heads, cfg.n_layers, cfg.n_layers,
            cfg.d_ff, cfg.dropout, batch_first=True, norm_first=True
        )
        self.head = nn.Linear(cfg.d_model, cfg.vocab, bias=False)
        self.dec_emb.weight = self.head.weight

    def forward(self, src: torch.Tensor, tgt: torch.Tensor,
                src_key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        T = tgt.size(1)
        causal = nn.Transformer.generate_square_subsequent_mask(T, device=src.device)
        enc_out = self.transformer.encoder(
            self.enc_pos(self.enc_emb(src)),
            src_key_padding_mask=src_key_padding_mask
        )
        dec_out = self.transformer.decoder(
            self.dec_pos(self.dec_emb(tgt)),
            enc_out,
            tgt_mask=causal,
            tgt_is_causal=True,
            memory_key_padding_mask=src_key_padding_mask
        )
        return self.head(dec_out)

    @torch.no_grad()
    def generate(self, prefix_ids: List[int], max_new: int = 64,
                 temperature: float = 0.7, top_k: int = 40,
                 tokenizer: Optional[CodeTokenizer] = None) -> List[int]:
        self.eval()
        dev = next(self.parameters()).device
        src = torch.tensor([prefix_ids], dtype=torch.long, device=dev)
        dec_ids = [SPECIAL["<BOS>"]]
        out_ids = []
        for _ in range(max_new):
            tgt = torch.tensor([dec_ids], dtype=torch.long, device=dev)
            logits = self(src, tgt)[0, -1] / temperature
            if top_k:
                topk_v, _ = torch.topk(logits, top_k)
                logits[logits < topk_v[-1]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            if nxt == SPECIAL["<EOS>"]:
                break
            dec_ids.append(nxt)
            out_ids.append(nxt)
        return out_ids


# ─────────────────────────────────────────────────────────────
# 4.  METRICS & VISUALISATION
# ─────────────────────────────────────────────────────────────

@dataclass
class MetricLog:
    train_loss:  List[float] = field(default_factory=list)
    val_loss:    List[float] = field(default_factory=list)
    train_ppl:   List[float] = field(default_factory=list)
    val_ppl:     List[float] = field(default_factory=list)
    lr:          List[float] = field(default_factory=list)
    token_acc:   List[float] = field(default_factory=list)   # top-1 accuracy
    grad_norm:   List[float] = field(default_factory=list)

    def append(self, **kw):
        for k, v in kw.items():
            getattr(self, k).append(v)


def plot_metrics(log: MetricLog, title: str, save_path: str):
    """
    Rich 2×3 dashboard saved to PNG.
    """
    epochs = list(range(1, len(log.train_loss) + 1))
    fig = plt.figure(figsize=(18, 10), facecolor="#0d1117")
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    ACCENT = "#58a6ff"
    WARN   = "#f78166"
    OK     = "#3fb950"
    GRID   = "#21262d"
    TXT    = "#c9d1d9"

    plt.rcParams.update({
        "axes.facecolor":   "#161b22",
        "axes.edgecolor":   GRID,
        "axes.labelcolor":  TXT,
        "xtick.color":      TXT,
        "ytick.color":      TXT,
        "text.color":       TXT,
        "grid.color":       GRID,
        "grid.linewidth":   0.6,
    })

    def _ax(pos, ylabel, title_s):
        ax = fig.add_subplot(pos)
        ax.set_xlabel("Epoch", fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(title_s, fontsize=10, color=ACCENT, pad=6)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(True)
        return ax

    # 1 — Loss
    ax = _ax(gs[0, 0], "Loss", "Train / Val Loss")
    ax.plot(epochs, log.train_loss, color=ACCENT, lw=1.8, label="train")
    if log.val_loss:
        ax.plot(epochs, log.val_loss, color=WARN, lw=1.8, linestyle="--", label="val")
    ax.legend(fontsize=8)

    # 2 — Perplexity
    ax = _ax(gs[0, 1], "Perplexity", "Train / Val Perplexity")
    ax.plot(epochs, log.train_ppl, color=ACCENT, lw=1.8, label="train")
    if log.val_ppl:
        ax.plot(epochs, log.val_ppl, color=WARN, lw=1.8, linestyle="--", label="val")
    ax.set_yscale("log")
    ax.legend(fontsize=8)

    # 3 — Token top-1 accuracy
    ax = _ax(gs[0, 2], "Accuracy", "Top-1 Token Accuracy")
    ax.plot(epochs, log.token_acc, color=OK, lw=1.8)
    ax.set_ylim(0, 1)

    # 4 — LR schedule
    ax = _ax(gs[1, 0], "LR", "Learning Rate")
    ax.plot(epochs, log.lr, color="#d2a8ff", lw=1.5)
    ax.set_yscale("log")

    # 5 — Gradient norm
    ax = _ax(gs[1, 1], "Grad Norm", "Gradient Norm")
    ax.plot(epochs, log.grad_norm, color="#ffa657", lw=1.5)

    # 6 — Train vs Val gap (over-fit indicator)
    ax = _ax(gs[1, 2], "Δ Loss (train-val)", "Generalisation Gap")
    if log.val_loss:
        gap = [v - t for t, v in zip(log.train_loss, log.val_loss)]
        ax.fill_between(epochs, 0, gap,
                        where=[g > 0 for g in gap], color=WARN, alpha=0.35, label="overfit")
        ax.fill_between(epochs, 0, gap,
                        where=[g <= 0 for g in gap], color=OK, alpha=0.35, label="underfit")
        ax.plot(epochs, gap, color=TXT, lw=1.0)
        ax.axhline(0, color=GRID, lw=1)
        ax.legend(fontsize=8)

    fig.suptitle(title, fontsize=14, color=ACCENT, y=1.01)
    plt.savefig(save_path, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"[Plot] saved → {save_path}")


# ─────────────────────────────────────────────────────────────
# 5.  CHECKPOINTING
# ─────────────────────────────────────────────────────────────



# ─────────────────────────────────────────────────────────────
# 6.  TRAINING LOOPS
# ─────────────────────────────────────────────────────────────

def _clip_norm(model: nn.Module, max_norm: float = 1.0) -> float:
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_token_model(
    model:      TokenModel,
    train_dl:   DataLoader,
    val_dl:     DataLoader,
    epochs:     int,
    lr:         float,
    device:     torch.device,
    saver:      BestModelSaver,
    log:        MetricLog,
    plot_dir:   str,
):
    one_part = len(train_dl) // 100
    print(f"Length of train_dl: {len(train_dl)}, 1% of which is {one_part}")
    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 20)
    crit = nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"])

    for ep in range(1, epochs + 1):
        # ── train ────────────────────────────────────────────
        print(f"Epoch {ep}")
        model.train()
        t_loss = t_acc = t_steps = 0
        for x, y in tqdm(train_dl, desc=f"[Token] Epoch {ep}/{epochs} train",
                 leave=False, unit="batch"):
            # if index % one_part == 0:
                # print(f"{index // one_part}% at {datetime.now().time()}")
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = crit(logits.view(-1, logits.size(-1)), y.view(-1))
            opt.zero_grad()
            loss.backward()
            gn = _clip_norm(model)
            opt.step()
            t_loss += loss.item()
            preds = logits.argmax(-1)
            mask = (y != SPECIAL["<PAD>"])
            t_acc += (preds[mask] == y[mask]).float().mean().item()
            t_steps += 1

        tl = t_loss / t_steps
        ta = t_acc / t_steps

        # ── val ──────────────────────────────────────────────
        model.eval()
        v_loss = v_steps = 0
        with torch.no_grad():
            for x, y in tqdm(val_dl, desc=f"[Token] Epoch {ep}/{epochs} val  ",
                 leave=False, unit="batch"):
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss   = crit(logits.view(-1, logits.size(-1)), y.view(-1))
                v_loss  += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl
        sched.step()

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"],
                   token_acc=ta, grad_norm=gn)

        print(f"[Token ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
              f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}  lr={opt.param_groups[0]['lr']:.2e}")

        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"Token Model — Epoch {ep}", f"{plot_dir}/token_metrics_ep{ep:03d}.png")

    plot_metrics(log, "Token Model — Final", f"{plot_dir}/token_metrics_final.png")


def train_line_model(
    model: LineModel,
    train_dl: DataLoader,
    val_dl: DataLoader,
    epochs: int,
    lr:  float,
    device: torch.device,
    saver: BestModelSaver,
    log: MetricLog,
    plot_dir: str,
        ):
    one_part = len(train_dl) // 100
    print(f"Length of train_dl: {len(train_dl)}, 1% of which is {one_part}")
    opt = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 20)
    crit = nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"])

    for ep in range(1, epochs + 1):
        print(f"Epoch {ep}")
        model.train()
        t_loss = t_acc = t_steps = 0
        for src, tgt in tqdm(train_dl, desc=f"[Token] Epoch {ep}/{epochs} train",
                 leave=False, unit="batch"):
            # if index % one_part == 0:
                # print(f"{index // one_part}% at {datetime.now().time()}")
            src, tgt = src.to(device), tgt.to(device)
            pad_mask = (src == SPECIAL["<PAD>"])
            dec_in = tgt[:, :-1]
            dec_out = tgt[:, 1:]
            logits = model(src, dec_in, src_key_padding_mask=pad_mask)
            loss = crit(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
            opt.zero_grad()
            loss.backward()
            gn = _clip_norm(model)
            opt.step()
            t_loss += loss.item()
            preds  = logits.argmax(-1)
            mask  = (dec_out != SPECIAL["<PAD>"])
            t_acc += (preds[mask] == dec_out[mask]).float().mean().item()
            t_steps += 1

        tl = t_loss / t_steps
        ta = t_acc  / t_steps

        model.eval()
        v_loss = v_steps = 0
        with torch.no_grad():
            for src, tgt in tqdm(val_dl, desc=f"[Token] Epoch {ep}/{epochs} val  ",
                 leave=False, unit="batch"):
                src, tgt = src.to(device), tgt.to(device)
                pad_mask = (src == SPECIAL["<PAD>"])
                dec_in   = tgt[:, :-1]
                dec_out  = tgt[:, 1:]
                logits   = model(src, dec_in, src_key_padding_mask=pad_mask)
                loss     = crit(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
                v_loss  += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl
        sched.step()

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"],
                   token_acc=ta, grad_norm=gn)

        print(f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
              f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}  lr={opt.param_groups[0]['lr']:.2e}")

        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"Line Model — Epoch {ep}", f"{plot_dir}/line_metrics_ep{ep:03d}.png")

    plot_metrics(log, "Line Model — Final", f"{plot_dir}/line_metrics_final.png")


# ─────────────────────────────────────────────────────────────
# 7.  INTERACTIVE HAND-TEST REPL
# ─────────────────────────────────────────────────────────────

def hand_test_repl(token_model: TokenModel, line_model: LineModel,
                   tokenizer: CodeTokenizer, device: torch.device):
    """
    Interactive REPL for testing both models.
    Commands:
      :token  <code prefix>   →  next-token completion
      :line   <code prefix>   →  full line completion
      :temp   <float>         →  set temperature
      :k      <int>           →  set top-k
      :quit                   →  exit
    """
    print("\n" + "═"*60)
    print("  Python Autocomplete — Interactive Test")
    print("  Commands: :token <prefix>  |  :line <prefix>")
    print("            :temp <float>   |  :k <int>  |  :quit")
    print("═"*60 + "\n")

    temperature = 0.8
    top_k       = 50

    while True:
        try:
            raw = input(">> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nBye!")
            break

        if not raw:
            continue

        if raw.startswith(":quit"):
            break
        elif raw.startswith(":temp"):
            try:   temperature = float(raw.split()[1])
            except: print("Usage: :temp 0.7")
            print(f"temperature = {temperature}")
            continue
        elif raw.startswith(":k"):
            try:   top_k = int(raw.split()[1])
            except: print("Usage: :k 40")
            print(f"top_k = {top_k}")
            continue
        elif raw.startswith(":token"):
            prefix = raw[6:].strip()
            ids    = tokenizer.encode(prefix)[:-1]   # drop EOS
            with torch.no_grad():
                new_ids = token_model.generate(
                    ids, max_new=20, temperature=temperature,
                    top_k=top_k, stop_at_word_end=True, tokenizer=tokenizer)
            completion = tokenizer.decode(new_ids)
            print(f"  ← token completion: {prefix}\033[32m{completion}\033[0m\n")
        elif raw.startswith(":line"):
            prefix = raw[5:].strip()
            ids    = tokenizer.encode(prefix)[:-1]
            with torch.no_grad():
                new_ids = line_model.generate(
                    ids, max_new=64, temperature=temperature,
                    top_k=top_k, tokenizer=tokenizer)
            completion = tokenizer.decode(new_ids)
            print(f"  ← line  completion: {prefix}\033[33m{completion}\033[0m\n")
        else:
            # default to line completion
            ids = tokenizer.encode(raw)[:-1]
            with torch.no_grad():
                new_ids = line_model.generate(
                    ids, max_new=64, temperature=temperature,
                    top_k=top_k, tokenizer=tokenizer)
            completion = tokenizer.decode(new_ids)
            print(f"  ← line  completion: {raw}\033[33m{completion}\033[0m\n")




# ─────────────────────────────────────────────────────────────
# 8.  MAIN
# ─────────────────────────────────────────────────────────────
class Arguments():
    def __init__(self, data_dir: str = "Clean_Dataset", ckpt_dir: str = "checkpoints",
                    plot_dir: str = "plots", tokenizer: str = "tokenizer.json", 
                    epochs: int = 5, batch: int = 32, lr: float = 5e-4,
                    ctx: int = 128, d_model: int = 256, n_layers: int = 4,
                    n_heads: int = 8, vocab_size: int = 000, max_files: int = 0,
                    val_split: float = 0.1, seed: int = 42, for_usage: bool = False,
                    skip_token: bool = False, skip_line: bool = False, test: bool = False):
        self.data_dir = data_dir
        self.ckpt_dir = ckpt_dir
        self.plot_dir = plot_dir
        self.tokenizer = tokenizer
        self.epochs = epochs
        self.batch = batch
        self.lr = lr
        self.ctx = ctx
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.vocab_size = vocab_size
        self.max_files = max_files
        self.val_split = val_split
        self.seed = seed
        self.skip_token = skip_token
        self.skip_line = skip_line
        self.test = test
        self.for_usage = for_usage


def main():
    # args = Arguments()
    # args = Arguments(max_files=100)
    # args = Arguments(skip_line=True, max_files=100, epochs=1)
    # args = Arguments(max_files=100, epochs=2, skip_token=True, vocab_size=160000)
    args = Arguments(skip_token=True)
    # args = Arguments(test=True)
    # args = Arguments(for_usage==True)
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir,  exist_ok=True)

    # ── tokenizer ────────────────────────────────────────────
    if os.path.exists(args.tokenizer):
        print(f"[Tokenizer] loading {args.tokenizer}")
        tokenizer = CodeTokenizer.load(args.tokenizer)
    else:
        print("[Tokenizer] building from data …")
        texts = load_files(args.data_dir, args.max_files)
        tokenizer = CodeTokenizer(vocab_size=args.vocab_size)
        tokenizer.build(texts)
        tokenizer.save(args.tokenizer)

    cfg = ModelCfg(
        vocab=tokenizer.vocab, d_model=args.d_model,
        n_heads=args.n_heads,  n_layers=args.n_layers,
        d_ff=args.d_model * 4, max_len=args.ctx + 32,
    )

    torch.serialization.add_safe_globals([ModelCfg])

    # ── test-only mode ───────────────────────────────────────
    if args.test:
        tok_saver  = BestModelSaver(args.ckpt_dir, TOKEN_MODEL_NAME)
        line_saver = BestModelSaver(args.ckpt_dir, LINE_MODEL_NAME)
        tm = TokenModel(cfg).to(device)
        lm = LineModel(cfg).to(device)
        for sav, model, name in [(tok_saver, tm, "token"), (line_saver, lm, "line")]:
            # find best ckpt by scanning dir
            paths = sorted(glob.glob(str(Path(args.ckpt_dir) / f"{name}_model_*.pt")))
            if paths:
                ck = torch.load(paths[0], map_location=device)
                model.load_state_dict(ck["model_state"])
                print(f"[Loaded] {name} from {paths[0]}")
        hand_test_repl(tm, lm, tokenizer, device)
        return
    
    
    # ──usage-only mode ───────────────────────────────────────
    if args.for_usage:
        tok_saver  = BestModelSaver(args.ckpt_dir, TOKEN_MODEL_NAME)
        line_saver = BestModelSaver(args.ckpt_dir, LINE_MODEL_NAME)
        tm = TokenModel(cfg).to(device)
        lm = LineModel(cfg).to(device)
        for sav, model, name in [(tok_saver, tm, "token"), (line_saver, lm, "line")]:
            # find best ckpt by scanning dir
            paths = sorted(glob.glob(str(Path(args.ckpt_dir) / f"{name}_model_*.pt")))
            if paths:
                ck = torch.load(paths[0], map_location=device)
                model.load_state_dict(ck["model_state"])
                print(f"[Loaded] {name} from {paths[0]}")
        hand_test_repl(tm, lm, tokenizer, device)
        return
    


    # ── load data ────────────────────────────────────────────
    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts:
        print("[ERROR] no data files found. Please put .py files in --data_dir")
        return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt = texts[:split]
    va_txt = texts[split:]
    
    # ── TOKEN MODEL ──────────────────────────────────────────
    if not args.skip_token:
        print("  Prepairing TOKEN model")

        # flatten all train text → single id stream
        all_ids_tr = []
        for t in tr_txt:
            all_ids_tr.extend(tokenizer.encode(t))
        all_ids_va = []
        for t in va_txt:
            all_ids_va.extend(tokenizer.encode(t))

        tr_ds = TokenDataset(all_ids_tr, args.ctx)
        va_ds = TokenDataset(all_ids_va, args.ctx)
        tr_dl = DataLoader(tr_ds, args.batch, shuffle=True,  num_workers=0, pin_memory=True)
        va_dl = DataLoader(va_ds, args.batch, shuffle=False, num_workers=0, pin_memory=True)

        tok_model = TokenModel(cfg).to(device)
        n_params  = sum(p.numel() for p in tok_model.parameters() if p.requires_grad)
        print(f"[Token Model] {n_params/1e6:.2f}M parameters")

        tok_saver = BestModelSaver(args.ckpt_dir, TOKEN_MODEL_NAME)
        tok_log   = MetricLog()
        print("  Training TOKEN model")
        train_token_model(tok_model, tr_dl, va_dl, args.epochs, args.lr,
                          device, tok_saver, tok_log, args.plot_dir)
    else:
        tok_model = TokenModel(cfg).to(device)
        tok_saver = BestModelSaver(args.ckpt_dir, TOKEN_MODEL_NAME)
        paths = sorted(glob.glob(str(Path(args.ckpt_dir) / TOKEN_MODEL_NAME + "_*.pt")))
        if paths:
            ck = torch.load(paths[0], map_location=device)
            tok_model.load_state_dict(ck["model_state"])

    # ── LINE MODEL ───────────────────────────────────────────
    if not args.skip_line:
        print("  Prepairing LINE model")

        tr_line_ds = LineDataset(tr_txt, tokenizer)
        va_line_ds = LineDataset(va_txt, tokenizer)
        collate = lambda b: collate_line(b, tokenizer.pad_id) 
        tr_line_dl = DataLoader(tr_line_ds, args.batch, shuffle=True,
                                # num_workers=0, pin_memory=True)
                                collate_fn=collate, num_workers=0, pin_memory=True)
        va_line_dl = DataLoader(va_line_ds, args.batch, shuffle=False,
                                # num_workers=0, pin_memory=True)
                                collate_fn=collate, num_workers=0, pin_memory=True)

        line_model = LineModel(cfg).to(device)
        n_params = sum(p.numel() for p in line_model.parameters() if p.requires_grad)
        print(f"[Line  Model] {n_params/1e6:.2f}M parameters")

        line_saver = BestModelSaver(args.ckpt_dir, LINE_MODEL_NAME)
        line_log = MetricLog()
        print("  Training LINE model")
        train_line_model(line_model, tr_line_dl, va_line_dl, args.epochs, args.lr,
                         device, line_saver, line_log, args.plot_dir)
    else:
        line_model = LineModel(cfg).to(device)
        paths = sorted(glob.glob(str(Path(args.ckpt_dir) / LINE_MODEL_NAME + "_*.pt")))
        if paths:
            ck = torch.load(paths[0], map_location=device)
            line_model.load_state_dict(ck["model_state"])

    # ── interactive test ─────────────────────────────────────
    hand_test_repl(tok_model, line_model, tokenizer, device)


main()

No traceback available to show.


[Device] cuda
PyTorch версия: 2.6.0+cu124
CUDA доступна: True
Версия CUDA, под которую собран PyTorch: 12.4
Количество GPU: 1
[Tokenizer] loading tokenizer.json
[Loading] Started loading
[Data] loaded 12110 files from Clean_Dataset
[Loading] Ended loading
  Prepairing LINE model


C:\Programing\code_autocomplete\venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


1510901
[LineDataset] 1006473 samples
162923
[LineDataset] 108435 samples
[Line  Model] 7.41M parameters
  Training LINE model
Length of train_dl: 31453, 1% of which is 314
Epoch 1


[Line  ep   1] train_loss=1.4504  val_loss=0.9505  ppl=2.6  acc=0.592  lr=4.55e-04
[Saver] saved ckpt: checkpoints\line_model_ep001_loss0.9505.pt  (val_loss=0.9505)
[Plot] saved → plots/line_metrics_ep001.png
Epoch 2


[Line  ep   2] train_loss=0.9610  val_loss=0.8411  ppl=2.3  acc=0.726  lr=3.36e-04
[Saver] saved ckpt: checkpoints\line_model_ep002_loss0.8411.pt  (val_loss=0.8411)
[Plot] saved → plots/line_metrics_ep002.png
Epoch 3


[Line  ep   3] train_loss=0.8729  val_loss=0.7917  ppl=2.2  acc=0.749  lr=1.89e-04
[Saver] saved ckpt: checkpoints\line_model_ep003_loss0.7917.pt  (val_loss=0.7917)
[Plot] saved → plots/line_metrics_ep003.png
Epoch 4


[Line  ep   4] train_loss=0.8228  val_loss=0.7606  ppl=2.1  acc=0.762  lr=7.04e-05
[Saver] removed old ckpt: checkpoints\line_model_ep001_loss0.9505.pt
[Saver] saved ckpt: checkpoints\line_model_ep004_loss0.7606.pt  (val_loss=0.7606)
[Plot] saved → plots/line_metrics_ep004.png
Epoch 5


[Line  ep   5] train_loss=0.7914  val_loss=0.7393  ppl=2.1  acc=0.770  lr=2.50e-05
[Saver] removed old ckpt: checkpoints\line_model_ep002_loss0.8411.pt
[Saver] saved ckpt: checkpoints\line_model_ep005_loss0.7393.pt  (val_loss=0.7393)
[Plot] saved → plots/line_metrics_ep005.png
[Plot] saved → plots/line_metrics_final.png

════════════════════════════════════════════════════════════
  Python Autocomplete — Interactive Test
  Commands: :token <prefix>  |  :line <prefix>
            :temp <float>   |  :k <int>  |  :quit
════════════════════════════════════════════════════════════




## Model evaluation

In [ ]:
"""
evaluate.py — Offline evaluation of trained autocomplete models.

Metrics computed:
  Token model : top-1 / top-5 accuracy, perplexity, mean reciprocal rank
  Line  model : exact-match@1, prefix-match, BLEU-4, chrF, perplexity
  Both        : latency (ms / sample)

Results are saved to JSON and a summary PNG dashboard.
"""

import os, json, time, glob, math, argparse
from pathlib import Path
from typing import List, Tuple, Optional

import torch
import torch.nn.functional as F
import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# import models from train.py (same directory)
import sys
sys.path.insert(0, os.path.dirname(__file__))


# ─────────────────────────────────────────────────────────────
# BLEU / chrF helpers (no NLTK dependency)
# ─────────────────────────────────────────────────────────────

def ngrams(seq: list, n: int) -> dict:
    counts = {}
    for i in range(len(seq) - n + 1):
        g = tuple(seq[i: i+n])
        counts[g] = counts.get(g, 0) + 1
    return counts


def bleu4(ref: list, hyp: list) -> float:
    if len(hyp) == 0:
        return 0.0
    bp = min(1.0, math.exp(1 - len(ref) / max(1, len(hyp))))
    score = 0.0
    for n in range(1, 5):
        r_ng = ngrams(ref, n)
        h_ng = ngrams(hyp, n)
        clip = sum(min(h_ng[g], r_ng.get(g, 0)) for g in h_ng)
        tot  = max(1, sum(h_ng.values()))
        if clip == 0:
            return 0.0
        score += math.log(clip / tot)
    return bp * math.exp(score / 4)


def chrf(ref: str, hyp: str, n: int = 6) -> float:
    def char_ngrams(s, n):
        return ngrams(list(s), n)
    scores = []
    for i in range(1, n + 1):
        r = char_ngrams(ref, i)
        h = char_ngrams(hyp, i)
        prec = sum(min(h[g], r.get(g, 0)) for g in h) / max(1, sum(h.values()))
        rec  = sum(min(r[g], h.get(g, 0)) for g in r) / max(1, sum(r.values()))
        f    = 2 * prec * rec / max(1e-9, prec + rec)
        scores.append(f)
    return float(np.mean(scores)) if scores else 0.0


# ─────────────────────────────────────────────────────────────
# Evaluation routines
# ─────────────────────────────────────────────────────────────

@torch.no_grad()
def eval_token_model(model: TokenModel, dataset: TokenDataset,
                     device: torch.device, n_samples: int = 2000):
    model.eval()
    crit = torch.nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"], reduction="sum")
    total_loss = total_tokens = 0
    top1_correct = top5_correct = 0
    mrr_sum = 0.0
    latencies = []

    indices = list(range(min(n_samples, len(dataset))))
    np.random.shuffle(indices)

    for idx in indices:
        x, y = dataset[idx]
        x = x.unsqueeze(0).to(device)
        y = y.unsqueeze(0).to(device)

        t0 = time.perf_counter()
        logits = model(x)
        latencies.append((time.perf_counter() - t0) * 1000)

        loss = crit(logits.view(-1, logits.size(-1)), y.view(-1))
        n_tok = (y != SPECIAL["<PAD>"]).sum().item()
        total_loss   += loss.item()
        total_tokens += n_tok

        # last-position metrics
        last_logit = logits[0, -1]
        true_id    = y[0, -1].item()
        if true_id == SPECIAL["<PAD>"]:
            continue
        sorted_ids = last_logit.argsort(descending=True).tolist()
        rank = sorted_ids.index(true_id) + 1 if true_id in sorted_ids else len(sorted_ids)
        top1_correct += (rank == 1)
        top5_correct += (rank <= 5)
        mrr_sum      += 1.0 / rank

    n = len(indices)
    return {
        "perplexity":  math.exp(min(total_loss / max(1, total_tokens), 20)),
        "top1_acc":    top1_correct / n,
        "top5_acc":    top5_correct / n,
        "mrr":         mrr_sum / n,
        "latency_ms":  float(np.mean(latencies)),
    }


@torch.no_grad()
def eval_line_model(model: LineModel, dataset: LineDataset,
                    tokenizer: CodeTokenizer, device: torch.device,
                    n_samples: int = 500):
    model.eval()
    exact = prefix20 = 0
    bleu_scores = []
    chrf_scores = []
    latencies   = []

    indices = list(range(min(n_samples, len(dataset))))
    np.random.shuffle(indices)

    for idx in indices:
        prefix_ids, target_ids = dataset[idx]
        # strip EOS from target for comparison
        target_ids = [i for i in target_ids if i != SPECIAL["<EOS>"]]

        t0 = time.perf_counter()
        hyp_ids = model.generate(prefix_ids, max_new=64,
                                 temperature=1.0, top_k=1,
                                 tokenizer=tokenizer)
        latencies.append((time.perf_counter() - t0) * 1000)

        exact    += (hyp_ids == target_ids)
        prefix20 += (hyp_ids[:20] == target_ids[:20])
        bleu_scores.append(bleu4(target_ids, hyp_ids))
        ref_str = tokenizer.decode(target_ids)
        hyp_str = tokenizer.decode(hyp_ids)
        chrf_scores.append(chrf(ref_str, hyp_str))

    n = len(indices)
    return {
        "exact_match":    exact / n,
        "prefix20_match": prefix20 / n,
        "bleu4":          float(np.mean(bleu_scores)),
        "chrf":           float(np.mean(chrf_scores)),
        "latency_ms":     float(np.mean(latencies)),
    }


# ─────────────────────────────────────────────────────────────
# Dashboard plot
# ─────────────────────────────────────────────────────────────

def plot_eval(tok_res: dict, line_res: dict, out_path: str):
    DARK  = "#0d1117"; MID = "#161b22"; GRID = "#21262d"
    BLUE  = "#58a6ff"; GREEN = "#3fb950"; ORG = "#ffa657"; TXT = "#c9d1d9"

    plt.rcParams.update({
        "axes.facecolor": MID, "axes.edgecolor": GRID,
        "axes.labelcolor": TXT, "xtick.color": TXT,
        "ytick.color": TXT, "text.color": TXT, "grid.color": GRID,
    })

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=DARK)

    # Token model bar chart
    ax = axes[0]
    metrics = ["top1_acc", "top5_acc", "mrr"]
    values  = [tok_res[m] for m in metrics]
    labels  = ["Top-1 Acc", "Top-5 Acc", "MRR"]
    bars = ax.bar(labels, values, color=[BLUE, GREEN, ORG], width=0.5, zorder=3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f"{val:.3f}",
                ha="center", va="bottom", fontsize=10, color=TXT)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Token Model\nPerplexity: {tok_res['perplexity']:.1f}  "
                 f"Latency: {tok_res['latency_ms']:.1f}ms",
                 fontsize=10, color=BLUE, pad=8)
    ax.grid(True, axis="y", lw=0.5, zorder=0)

    # Line model bar chart
    ax = axes[1]
    metrics2 = ["exact_match", "prefix20_match", "bleu4", "chrf"]
    values2  = [line_res[m] for m in metrics2]
    labels2  = ["Exact\nMatch", "Prefix-20\nMatch", "BLEU-4", "chrF"]
    bars2 = ax.bar(labels2, values2, color=[BLUE, GREEN, ORG, "#d2a8ff"], width=0.5, zorder=3)
    for bar, val in zip(bars2, values2):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f"{val:.3f}",
                ha="center", va="bottom", fontsize=10, color=TXT)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Line Model\nLatency: {line_res['latency_ms']:.1f}ms",
                 fontsize=10, color=BLUE, pad=8)
    ax.grid(True, axis="y", lw=0.5, zorder=0)

    fig.suptitle("Evaluation Dashboard", fontsize=14, color=BLUE)
    plt.tight_layout()
    plt.savefig(out_path, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"[Eval] plot → {out_path}")


# ─────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data_dir",   default="data/val")
    p.add_argument("--ckpt_dir",   default="checkpoints")
    p.add_argument("--tokenizer",  default="tokenizer.json")
    p.add_argument("--out_dir",    default="eval_results")
    p.add_argument("--n_tok",      type=int, default=2000)
    p.add_argument("--n_line",     type=int, default=500)
    p.add_argument("--ctx",        type=int, default=128)
    args = p.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = CodeTokenizer.load(args.tokenizer)

    # ── load best checkpoints ────────────────────────────────
    def load_best(prefix: str, model_cls, cfg):
        paths = sorted(glob.glob(str(Path(args.ckpt_dir) / f"{prefix}_*.pt")))
        if not paths:
            print(f"[Eval] No {prefix} checkpoint found in {args.ckpt_dir}")
            return None
        ck = torch.load(paths[0], map_location=device)
        m  = model_cls(cfg).to(device)
        m.load_state_dict(ck["model_state"])
        print(f"[Eval] loaded {prefix} from {paths[0]}  (val_loss={ck['val_loss']:.4f})")
        return m

    cfg = ModelCfg(vocab=tokenizer.vocab, d_model=256, n_heads=8,
                   n_layers=4, d_ff=1024, max_len=args.ctx+32)

    tok_model  = load_best(TOKEN_MODEL_NAME, TokenModel, cfg)
    line_model = load_best(LINE_MODEL_NAME, LineModel, cfg)

    texts = load_files(args.data_dir)
    if not texts:
        print("[Eval] no data found — run prepare_data.py first"); return

    all_ids = []
    for t in texts:
        all_ids.extend(tokenizer.encode(t))

    results = {}

    if tok_model:
        print("[Eval] evaluating Token model …")
        ds  = TokenDataset(all_ids, args.ctx)
        res = eval_token_model(tok_model, ds, device, args.n_tok)
        results["token"] = res
        print(json.dumps(res, indent=2))

    if line_model:
        print("[Eval] evaluating Line model …")
        ds  = LineDataset(texts, tokenizer)
        res = eval_line_model(line_model, ds, tokenizer, device, args.n_line)
        results["line"] = res
        print(json.dumps(res, indent=2))

    out_json = os.path.join(args.out_dir, "eval_results.json")
    with open(out_json, "w") as f:
        json.dump(results, f, indent=2)
    print(f"[Eval] results → {out_json}")

    if tok_model and line_model:
        plot_eval(results["token"], results["line"],
                  os.path.join(args.out_dir, "eval_dashboard.png"))


main()